# Credit Risk — Logistic Regression, LightGBM and inference (v3.1)

**Trương Minh Hoàng · MSSV 25410008**

**ARCHIVED_SOURCE — source review, not a fresh benchmark.** Bản học tập lịch sử được giữ để giải thích quá trình làm bài. Output cũ và thông tin môi trường riêng đã được bỏ; xem README/AUDIT của dự án trước khi chạy. Historical calculations and known limitations are documented separately.

<a href="https://colab.research.google.com/github/hoangtm3979/credit/blob/main/B%E1%BA%A3n_V3_1_deploy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **✅  Đề tài 15: Ứng dụng AI đánh giá điểm tín dụng cá nhân  trong Fintech & Banking (Credit Scoring)  (v3.1)**



```
# 🧱 0) Cài đặt & Import
```



In [ ]:
# =========================================================
# FINAL v3.1: Credit Scoring (Historical pipeline; see AUDIT.md before reuse)
# =========================================================

# 0) Mount & Install
# Optional on Colab: mount your own Drive before setting CREDIT_DATA_DIR.

# Install research/requirements.txt before running.

# 1) Imports
import os, json, warnings, gc
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    brier_score_loss, precision_score, recall_score, f1_score
)
from sklearn.calibration import calibration_curve
from sklearn import __version__ as skver
import joblib

import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
import shap

📁 1) Đường dẫn, Load & Nhãn

In [ ]:
# 2) Paths
DATA_DIR = os.environ.get("CREDIT_DATA_DIR", "data/local")
TRAIN_PATH = os.path.join(DATA_DIR, "loan_2014_18.csv")
VALID_PATH = os.path.join(DATA_DIR, "loan_2019_20.csv")
OUT_DIR = os.environ.get("CREDIT_OUTPUT_DIR", "artifacts/credit-v3.1")
os.makedirs(OUT_DIR, exist_ok=True)

# 3) Load
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
valid_df = pd.read_csv(VALID_PATH, low_memory=False)
print("Train:", train_df.shape, "Valid:", valid_df.shape)

# 4) Make Label (target)
def label_encode(status: str):
    s = str(status).lower()
    return 1 if ("charged off" in s or "default" in s or "late" in s) else 0

if "loan_status" in train_df.columns:
    train_df["target"] = train_df["loan_status"].apply(label_encode)
    valid_df["target"] = valid_df["loan_status"].apply(label_encode)
elif "default_ind" in train_df.columns:
    train_df["target"] = train_df["default_ind"].astype(int)
    valid_df["target"] = valid_df["default_ind"].astype(int)
else:
    raise ValueError("Không tìm thấy cột label (loan_status hoặc default_ind).")

print("Label dist (train):", train_df["target"].value_counts(normalize=True).to_dict())


🧹 2) Làm sạch cơ bản

In [ ]:
# 5) Basic cleaning
DROP_COLS = ['id','member_id','Unnamed: 0']
train_df = train_df.drop(columns=[c for c in DROP_COLS if c in train_df.columns], errors='ignore')
valid_df = valid_df.drop(columns=[c for c in DROP_COLS if c in valid_df.columns], errors='ignore')

def basic_impute(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.columns:
        if c == "target":
            continue
        if df[c].dtype == "object":
            mode = df[c].mode(dropna=True)
            df[c] = df[c].fillna(mode.iloc[0] if len(mode) else "unknown")
        else:
            df[c] = df[c].fillna(df[c].median())
    return df

train_df = basic_impute(train_df)
valid_df = basic_impute(valid_df)


🧮 3) CIC Features

In [ ]:
# 6) CIC Feature Engineering (safe)
def add_cic_features(df):
    if 'mths_since_last_delinq' in df.columns:
        df['payment_history_score'] = df['mths_since_last_delinq'].fillna(999).clip(lower=0, upper=999)
    if {'loan_amnt','annual_inc'}.issubset(df.columns):
        df['credit_utilization'] = df['loan_amnt'] / (df['annual_inc'] + 1.0)
    if 'earliest_cr_line' in df.columns:
        parsed = pd.to_datetime(df['earliest_cr_line'], errors='coerce')
        df['credit_history_length'] = ((pd.to_datetime("today") - parsed).dt.days // 30).clip(lower=0)
    if {'open_acc','total_acc'}.issubset(df.columns):
        denom = df['total_acc'].replace(0, np.nan)
        df['credit_mix'] = (df['open_acc'] / denom).fillna(0.0)
    if 'inq_last_6mths' in df.columns:
        df['new_credit'] = df['inq_last_6mths']
    return df

train_df = add_cic_features(train_df)
valid_df = add_cic_features(valid_df)
print(f"Train (after CIC feats): {train_df.shape}  Valid: {valid_df.shape}")

# Split
y_train = train_df['target'].astype(int)
X_train = train_df.drop(columns=['target'])
y_test  = valid_df['target'].astype(int)
X_test  = valid_df.drop(columns=['target'])


🛡️ 4) Chống Leakage

In [ ]:
LEAK_EXACT = [
    "loan_status","issue_d","url","pymnt_plan",
    "collection_recovery_fee","recoveries",
    "debt_settlement_flag","debt_settlement_flag_date",
    "settlement_status","settlement_date","settlement_amount",
    "settlement_percentage","settlement_term",
    "orig_projected_additional_accrued_interest",
    "hardship_flag","hardship_type","hardship_reason","hardship_status",
    "hardship_start_date","hardship_end_date","hardship_loan_status",
    "hardship_payoff_balance_amount","hardship_last_payment_amount",
    "last_pymnt_amnt","total_pymnt","total_pymnt_inv",
    "total_rec_prncp","total_rec_int","total_rec_late_fee",
    "out_prncp","out_prncp_inv",
    "last_pymnt_d","next_pymnt_d","last_credit_pull_d",
    "payment_plan_start_date"
]
LEAK_PREFIX = ["out_", "last_pymnt_", "next_pymnt_", "debt_settlement_", "hardship_", "settlement_"]

def drop_leakage(df):
    to_drop = set([c for c in LEAK_EXACT if c in df.columns])
    for p in LEAK_PREFIX:
        to_drop.update([c for c in df.columns if c.startswith(p)])
    return df.drop(columns=list(to_drop), errors="ignore")

X_train = drop_leakage(X_train)
X_test  = drop_leakage(X_test)


🔡 5) Chuẩn bị bản cho LightGBM (parse numeric, encode an toàn)

In [ ]:
X_train_lgb = X_train.copy()
X_test_lgb  = X_test.copy()

# Numeric-like converters
def to_float_percent(s):
    s=str(s); s2=s.replace('%','')
    try:
        return float(s2)
    except (TypeError, ValueError):
        return np.nan

def parse_term(s):
    s=str(s)
    digits = ''.join(ch for ch in s if ch.isdigit())
    try:
        return float(digits)
    except:
        return np.nan

def parse_emp_length(x):
    x=str(x).lower()
    if '10+' in x: return 10
    if '< 1' in x: return 0
    if 'n/a' in x: return -1
    for t in x.split():
        if t.isdigit(): return int(t)
    return -1

def safe_apply(df, col, func, dtype=float):
    if col in df.columns:
        df[col] = df[col].apply(func).astype(dtype, errors="ignore")

# parse numeric-like BEFORE label encoding
for col, func, dtype in [
    ("int_rate", to_float_percent, float),
    ("revol_util", to_float_percent, float),
    ("term", parse_term, float),
    ("emp_length", parse_emp_length, int),
]:
    safe_apply(X_train_lgb, col, func, dtype)
    safe_apply(X_test_lgb, col, func, dtype)

# Convert earliest_cr_line -> months
def datetime_to_months(df, col):
    if col in df.columns:
        parsed = pd.to_datetime(df[col], errors='coerce')
        df[col] = ((pd.to_datetime("today") - parsed).dt.days // 30).fillna(0).astype('int32')

datetime_to_months(X_train_lgb, "earliest_cr_line")
datetime_to_months(X_test_lgb , "earliest_cr_line")

# High-cardinality: count-encode if exists (AND save mapping)
COUNT_COLS = ["emp_title", "title", "zip_code"]
count_maps = {}
for col in COUNT_COLS:
    if col in X_train_lgb.columns:
        vc = X_train_lgb[col].astype(str).value_counts()
        count_maps[col] = vc.to_dict()  # save mapping for inference
        X_train_lgb[col + "_count"] = X_train_lgb[col].astype(str).map(vc).fillna(1).astype('int32')
        if col in X_test_lgb.columns:
            X_test_lgb[col + "_count"] = X_test_lgb[col].astype(str).map(vc).fillna(1).astype('int32')
        # drop raw column
        X_train_lgb.drop(columns=[col], inplace=True)
        if col in X_test_lgb.columns:
            X_test_lgb.drop(columns=[col], inplace=True)

# Label-encode remaining object cols with safe mapping (unknown -> -1) & SAVE mapping
def fit_safe_label_encoders(train_df: pd.DataFrame, cols):
    encs = {}
    for c in cols:
        uniq = pd.Series(train_df[c].astype(str).unique())
        mapping = {v:i for i,v in enumerate(uniq)}
        encs[c] = mapping
    return encs

def transform_with_encoders(df: pd.DataFrame, encs: dict):
    for c, mapping in encs.items():
        if c in df.columns:
            df[c] = df[c].astype(str).map(mapping).fillna(-1).astype('int32')
    return df

obj_cols_lgb = X_train_lgb.select_dtypes(include=['object']).columns.tolist()
label_encoders = fit_safe_label_encoders(X_train_lgb, obj_cols_lgb)
X_train_lgb = transform_with_encoders(X_train_lgb, label_encoders)
X_test_lgb  = transform_with_encoders(X_test_lgb,  label_encoders)

# Ensure numeric dtype & no NaNs
X_train_lgb = X_train_lgb.apply(pd.to_numeric, errors='ignore').fillna(0)
X_test_lgb  = X_test_lgb.apply(pd.to_numeric, errors='ignore').fillna(0)

# Save feature order for LGBM inference
lgb_feature_order = X_train_lgb.columns.tolist()


🧪 6) Logistic Baseline (OHE an toàn với version guard)

In [ ]:
# 7) LOGISTIC baseline
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

# Version guard cho OneHotEncoder
sk_major, sk_minor = map(int, skver.split(".")[:2])
if (sk_major, sk_minor) >= (1, 2):
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
else:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=True)

preprocess_dense = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(with_mean=False), num_cols),
        ('cat', ohe, cat_cols)
    ],
    remainder='drop'
)

Xtr_enc = preprocess_dense.fit_transform(X_train)
Xte_enc = preprocess_dense.transform(X_test)

logit = LogisticRegression(max_iter=500, class_weight="balanced", n_jobs=-1)
logit.fit(Xtr_enc, y_train)
y_pred_logit = logit.predict_proba(Xte_enc)[:,1]


🌲 7) LightGBM Train

In [ ]:
# 8) LGBM (class weight by scale_pos_weight)
pos = y_train.sum(); neg = len(y_train) - pos
scale_pos = neg / max(pos, 1)

lgbm = lgb.LGBMClassifier(
    n_estimators=600, learning_rate=0.05,
    max_depth=6, num_leaves=31,
    min_child_samples=1000,
    min_gain_to_split=0.1,
    feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=1,
    scale_pos_weight=scale_pos,
    n_jobs=-1
)
lgbm.fit(
    X_train_lgb, y_train,
    eval_set=[(X_test_lgb, y_test)],
    eval_metric="auc",
    callbacks=[log_evaluation(50), early_stopping(50)]
)
y_pred_lgbm = lgbm.predict_proba(X_test_lgb)[:,1]


📊 8) Đánh giá + PSI + Ảnh

In [ ]:
# 9) Evaluation
def ks_stat(y_true,y_pred):
    fpr,tpr,_ = roc_curve(y_true,y_pred)
    return np.max(tpr-fpr)

print(f"AUC Logistic : {roc_auc_score(y_test,y_pred_logit):.10f} | KS: {ks_stat(y_test,y_pred_logit):.10f}")
print(f"AUC LightGBM : {roc_auc_score(y_test,y_pred_lgbm):.10f} | KS: {ks_stat(y_test,y_pred_lgbm):.10f}")

def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins+1)
    idx = np.digitize(y_prob, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = (idx == b)
        if mask.sum()==0: continue
        conf = y_prob[mask].mean()
        acc  = y_true[mask].mean()
        ece += (mask.mean())*abs(acc-conf)
    return ece

def ks_threshold(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    i = np.argmax(tpr-fpr); return thr[i], (tpr[i]-fpr[i])

def youden_threshold(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    J = tpr - fpr; i = np.argmax(J)
    return thr[i], J[i]

for name, pred in {"Logistic": y_pred_logit, "LightGBM": y_pred_lgbm}.items():
    brier = brier_score_loss(y_test, pred)
    ece   = expected_calibration_error(y_test, pred, n_bins=10)
    thr_ks, ks_val = ks_threshold(y_test, pred)
    thr_y,  J_val  = youden_threshold(y_test, pred)
    def rpt(thr):
        y_hat=(pred>=thr).astype(int)
        return precision_score(y_test,y_hat), recall_score(y_test,y_hat), f1_score(y_test,y_hat)
    p_ks,r_ks,f1_ks = rpt(thr_ks)
    p_y ,r_y ,f1_y  = rpt(thr_y)
    print(f"\n[{name}] Brier={brier:.4f} | ECE={ece:.4f}")
    print(f"KS thr={thr_ks:.4f}, KS={ks_val:.4f} -> P={p_ks:.3f},R={r_ks:.3f},F1={f1_ks:.3f}")
    print(f"Youden thr={thr_y:.4f}, J={J_val:.4f} -> P={p_y:.3f},R={r_y:.3f},F1={f1_y:.3f}")

# 10) PSI (train prob vs test prob)
def calculate_psi(expected, actual, buckets=10, eps=1e-6):
    q = np.linspace(0,100,buckets+1)
    cuts = np.unique(np.percentile(expected, q))
    if cuts.size < 2:
        return 0.0
    exp_cnt,_ = np.histogram(expected, bins=cuts)
    act_cnt,_ = np.histogram(actual, bins=cuts)
    exp_pct = exp_cnt / max(len(expected),1)
    act_pct = act_cnt / max(len(actual),1)
    return float(np.sum((exp_pct-act_pct)*np.log((exp_pct+eps)/(act_pct+eps))))

psi = calculate_psi(lgbm.predict_proba(X_train_lgb)[:,1], y_pred_lgbm)
print(f"PSI (LGBM train vs test): {psi:.4f}")

# 11) Feature Importance
imp = pd.DataFrame({'feature':X_train_lgb.columns,'importance':lgbm.booster_.feature_importance()})
topk = imp.sort_values("importance",ascending=False).head(15)
print("\nTop 15 features:\n", topk)

# 12) Save results table
res_table = pd.DataFrame({
    "Model":["Logistic","LightGBM"],
    "AUC":[roc_auc_score(y_test,y_pred_logit),roc_auc_score(y_test,y_pred_lgbm)],
    "KS":[ks_stat(y_test,y_pred_logit),ks_stat(y_test,y_pred_lgbm)]
})
res_table.to_csv(os.path.join(OUT_DIR,"model_results.csv"),index=False)
print("✅ Saved model_results.csv")

# 13) SHAP + Plots
try:
    explainer = shap.TreeExplainer(lgbm)
    n_sample = min(800, len(X_test_lgb))
    sample_X = X_test_lgb.sample(n_sample, random_state=42)
    shap_values = explainer.shap_values(sample_X)
    shap.summary_plot(shap_values, sample_X, show=False)
    plt.savefig(os.path.join(OUT_DIR,"shap_summary.png")); plt.close()
    print("✅ Saved SHAP summary")
except Exception as e:
    print("SHAP skipped:", e)

def plot_roc():
    plt.figure()
    fpr,tpr,_=roc_curve(y_test,y_pred_logit)
    plt.plot(fpr,tpr,label=f"Logistic AUC={roc_auc_score(y_test,y_pred_logit):.4f}")
    fpr,tpr,_=roc_curve(y_test,y_pred_lgbm)
    plt.plot(fpr,tpr,label=f"LightGBM AUC={roc_auc_score(y_test,y_pred_lgbm):.4f}")
    plt.plot([0,1],[0,1],'--',c='gray'); plt.title("ROC Curve"); plt.legend()
    plt.savefig(os.path.join(OUT_DIR,"roc_curve.png")); plt.close()

def plot_pr():
    plt.figure()
    prec,rec,_=precision_recall_curve(y_test,y_pred_logit)
    plt.plot(rec,prec,label=f"Logistic AP={average_precision_score(y_test,y_pred_logit):.4f}")
    prec,rec,_=precision_recall_curve(y_test,y_pred_lgbm)
    plt.plot(rec,prec,label=f"LightGBM AP={average_precision_score(y_test,y_pred_lgbm):.4f}")
    plt.title("Precision-Recall Curve"); plt.legend()
    plt.savefig(os.path.join(OUT_DIR,"pr_curve.png")); plt.close()

def plot_calib():
    plt.figure()
    for name,pred in {"Logistic":y_pred_logit,"LightGBM":y_pred_lgbm}.items():
        prob_true,prob_pred=calibration_curve(y_test,pred,n_bins=10)
        plt.plot(prob_pred,prob_true,"-o",label=name)
    plt.plot([0,1],[0,1],'--',c='gray'); plt.title("Calibration Plot"); plt.legend()
    plt.savefig(os.path.join(OUT_DIR,"calibration_plot.png")); plt.close()

plot_roc(); plot_pr(); plot_calib()
print("✅ Saved ROC/PR/Calibration plots")


**# Xuất cho báo cáo**

In [ ]:
# =========================================================
# 8+) Dataset Analysis & Profiling (Fixed Version)
# =========================================================

# Gộp lại để profile (train + valid)
df_all = pd.concat([train_df, valid_df], axis=0, ignore_index=True)

# 1. Basic Info
print("Dataset shape:", df_all.shape)
print("Train:", X_train.shape, "Test:", X_test.shape)

# 2. Target distribution
target_dist = df_all["target"].value_counts(normalize=True)
print("\nTarget distribution:\n", target_dist)

# 3. Missing values summary
missing_summary = df_all.isna().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
if not missing_summary.empty:
    print("\nTop missing values:\n", missing_summary.head(20))
else:
    print("\nNo missing values detected.")

# 4. Numeric feature stats
desc_numeric = df_all.describe(include=[np.number]).T
print("\nNumeric stats:\n", desc_numeric.head())

# 5. Categorical feature stats (top 10 categories)
desc_categorical = {}
for col in df_all.select_dtypes(include=["object","category"]).columns:
    desc_categorical[col] = df_all[col].value_counts(normalize=True).head(10)
print("\nCategorical stats (top categories):")
for k,v in desc_categorical.items():
    print(f"- {k}:\n{v}\n")

# 6. Correlation heatmap (for numeric features)
import seaborn as sns
plt.figure(figsize=(10,8))
corr = df_all.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Numeric Features)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"corr_heatmap.png"))
plt.close()

# 7. Class balance plot
plt.figure(figsize=(4,4))
target_dist.plot(kind="bar", color=["steelblue","salmon"])
plt.title("Target Distribution")
plt.savefig(os.path.join(OUT_DIR,"target_dist.png"))
plt.close()

print("✅ Saved dataset profiling (correlation heatmap, target distribution)")


🧾 9) Xuất Artifacts (để deploy)

In [ ]:
# 14) Export artifacts (models + preprocessors + encoders + feature order + count maps)

# LightGBM booster + feature order
lgbm.booster_.save_model(os.path.join(OUT_DIR, "lgbm_model.txt"))
with open(os.path.join(OUT_DIR, "lgbm_feature_order.json"), "w") as f:
    json.dump(lgb_feature_order, f)

# Logistic + preprocess pipeline
joblib.dump(logit, os.path.join(OUT_DIR, "logit_model.pkl"))
joblib.dump(preprocess_dense, os.path.join(OUT_DIR, "preprocess_dense.pkl"))
with open(os.path.join(OUT_DIR, "logit_columns.json"), "w") as f:
    json.dump({"num_cols": num_cols, "cat_cols": cat_cols, "all_cols": X_train.columns.tolist()}, f)

# Label encoders mapping (safe mapping dict)
with open(os.path.join(OUT_DIR, "label_encoders.json"), "w") as f:
    json.dump(label_encoders, f)

# Count-encode maps (emp_title/title/zip_code)
with open(os.path.join(OUT_DIR, "count_maps.json"), "w") as f:
    json.dump(count_maps, f)

print("✅ Exported: LGBM booster, feature_order, Logistic, preprocess_dense, label_encoders, count_maps")


📈 10) Mapping CIC + Advice (giữ như V3)

In [ ]:
# 15) CIC Mapping + Advice
def map_score_to_rank(score):
    mapping=[
        (403,429,"Hạng 10","Xấu"),
        (430,454,"Hạng 09","Dưới TB"),
        (455,479,"Hạng 08","Trung bình"),
        (480,544,"Hạng 07","Trung bình"),
        (545,571,"Hạng 06","Trung bình"),
        (572,587,"Hạng 05","Tốt"),
        (588,605,"Hạng 04","Tốt"),
        (606,621,"Hạng 03","Tốt"),
        (622,644,"Hạng 02","Rất tốt"),
        (645,706,"Hạng 01","Rất tốt")]
    for lo,hi,rank,label in mapping:
        if lo<=score<=hi: return rank,label
    return "N/A","N/A"

def prob_to_cic(prob):
    return 403 + (706-403)*(1-prob)

example_p = float(y_pred_lgbm[0])
example_score = prob_to_cic(example_p)
rank,label = map_score_to_rank(example_score)
print(f"Example -> prob_bad: {example_p:.6f}  score: {example_score:.2f}  rank: {rank}  label: {label}")

def give_advice(x_row, prob):
    advice=[]
    cu = x_row.get("credit_utilization", 0.0)
    ph = x_row.get("payment_history_score", 999)
    chl = x_row.get("credit_history_length", 0)
    oa  = x_row.get("open_acc", 0)
    if prob>0.5: advice.append("Thanh toán đúng hạn để cải thiện lịch sử tín dụng.")
    if cu>0.7: advice.append("Giảm tỷ lệ nợ/tổng thu nhập xuống dưới 70%.")
    if ph<12: advice.append("Duy trì 12 tháng không trễ hạn để nâng điểm.")
    if chl<24: advice.append("Duy trì lâu hơn tài khoản tín dụng để tăng uy tín.")
    if oa>10: advice.append("Không mở thêm quá nhiều tài khoản tín dụng mới.")
    if len(advice)==0: advice.append("Duy trì thói quen tốt để giữ điểm cao.")
    return advice

sample_case = X_test.iloc[0].to_dict()
advice = give_advice(sample_case, example_p)
print("✅ Gợi ý cải thiện:", advice)


🚀 11) Inference (Web/API) – dùng artifacts (không train lại)

In [ ]:
# inference.py (ví dụ)
import json, joblib, numpy as np, pandas as pd, lightgbm as lgb

# === Load artifacts ===
OUT_DIR = os.environ.get("CREDIT_OUTPUT_DIR", "artifacts/credit-v3.1")
booster = lgb.Booster(model_file=os.path.join(OUT_DIR,"lgbm_model.txt"))
preprocess_dense = joblib.load(os.path.join(OUT_DIR,"preprocess_dense.pkl"))
logit = joblib.load(os.path.join(OUT_DIR,"logit_model.pkl"))

with open(os.path.join(OUT_DIR,"lgbm_feature_order.json")) as f:
    lgb_feature_order = json.load(f)
with open(os.path.join(OUT_DIR,"label_encoders.json")) as f:
    label_encoders = json.load(f)
with open(os.path.join(OUT_DIR,"count_maps.json")) as f:
    count_maps = json.load(f)
with open(os.path.join(OUT_DIR,"logit_columns.json")) as f:
    logit_cols = json.load(f)
num_cols, cat_cols, all_cols = logit_cols["num_cols"], logit_cols["cat_cols"], logit_cols["all_cols"]

# === Preprocessing helpers (must match training) ===
def to_float_percent(s):
    try:
        return float(str(s).replace('%',''))
    except:
        return np.nan

def parse_term(s):
    s=str(s)
    digits = ''.join(ch for ch in s if ch.isdigit())
    try:
        return float(digits)
    except:
        return np.nan

def parse_emp_length(x):
    x=str(x).lower()
    if '10+' in x: return 10
    if '< 1' in x: return 0
    if 'n/a' in x: return -1
    for t in x.split():
        if t.isdigit(): return int(t)
    return -1

def datetime_to_months_single(s):
    parsed = pd.to_datetime(s, errors='coerce')
    if pd.isna(parsed): return 0
    return int(((pd.to_datetime("today") - parsed).days) // 30)

def apply_count_encode(val, mapping):
    return int(mapping.get(str(val), 1))

def transform_for_lgbm(x: dict) -> pd.DataFrame:
    # x: raw dict input
    rec = dict(x)  # copy
    # parse numeric-like
    if "int_rate" in rec: rec["int_rate"] = to_float_percent(rec["int_rate"])
    if "revol_util" in rec: rec["revol_util"] = to_float_percent(rec["revol_util"])
    if "term" in rec: rec["term"] = parse_term(rec["term"])
    if "emp_length" in rec: rec["emp_length"] = parse_emp_length(rec["emp_length"])
    if "earliest_cr_line" in rec: rec["earliest_cr_line"] = datetime_to_months_single(rec["earliest_cr_line"])

    # count-encode
    for col in ["emp_title","title","zip_code"]:
        if col in rec:
            rec[col+"_count"] = apply_count_encode(rec[col], count_maps.get(col, {}))
            rec.pop(col, None)

    df = pd.DataFrame([rec])

    # label-encode remaining object cols using saved mapping
    for c, mapping in label_encoders.items():
        if c in df.columns:
            df[c] = df[c].astype(str).map(mapping).fillna(-1).astype('int32')

    # Fill missing columns & order
    for col in lgb_feature_order:
        if col not in df.columns:
            df[col] = 0
    df = df[lgb_feature_order].fillna(0)
    return df

def transform_for_logit(x: dict) -> pd.DataFrame:
    # dựng DataFrame với các cột như training (all_cols)
    rec = {c: x.get(c, np.nan) for c in all_cols}
    df = pd.DataFrame([rec])
    # fill numerics
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
    # fill cats
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype(str).fillna("unknown")
    return df

def prob_to_cic(prob):
    return 403 + (706-403)*(1-prob)

def map_score_to_rank(score):
    mapping=[
        (403,429,"Hạng 10","Xấu"),(430,454,"Hạng 09","Dưới TB"),
        (455,479,"Hạng 08","Trung bình"),(480,544,"Hạng 07","Trung bình"),
        (545,571,"Hạng 06","Trung bình"),(572,587,"Hạng 05","Tốt"),
        (588,605,"Hạng 04","Tốt"),(606,621,"Hạng 03","Tốt"),
        (622,644,"Hạng 02","Rất tốt"),(645,706,"Hạng 01","Rất tốt")]
    for lo,hi,rank,label in mapping:
        if lo<=score<=hi: return rank,label
    return "N/A","N/A"

def predict_one(x_raw: dict):
    # Primary: LGBM
    Xlgb = transform_for_lgbm(x_raw)
    p = float(booster.predict(Xlgb)[0])
    cic = prob_to_cic(p)
    rank, label = map_score_to_rank(cic)

    # Optional logistic (for checking / ensemble)
    Xlog = transform_for_logit(x_raw)
    Xlog_enc = preprocess_dense.transform(Xlog)
    p_logit = float(logit.predict_proba(Xlog_enc)[:,1][0])

    return {
        "prob_bad_lgbm": p,
        "prob_bad_logit": p_logit,
        "cic_score": cic,
        "cic_rank": rank,
        "cic_label": label
    }

# Example inference:
# sample = { ... truyền dict giống cột raw ... }
# print(predict_one(sample))

Lưu ý deploy: Không train lại. Chỉ load artifacts và dùng đúng pipeline transform_for_lgbm / transform_for_logit.

🧩 triển khai webapp/API

Streamlit: dựng form nhập dữ liệu khách hàng → gọi predict_one() → Hiển thị CIC score + rank + advice.

FastAPI: tạo POST /score nhận JSON → dùng inference pipeline trên → trả JSON.

Monitoring: cron job hàng tuần tính PSI dựa trên score production vs score train; cảnh báo nếu PSI ≥ 0.25.